# Chapter 2 - Lab 5: Financial News Agent with Evaluator-Optimizer Pattern

This lab implements an **Evaluator-Optimizer** pattern with two agents:

- **web_news_searcher** searches recent Reuters financial news with the OpenAI hosted web search tool.
- **news_evaluator** checks whether the result satisfies the original user request (number of items, dates, region/topic and direct Reuters links).

The search/evaluation loop stops when the evaluator marks the result as successful or when the maximum number of attempts is reached.


## 1. Install dependencies

Run this cell in a fresh Colab runtime. If Colab asks for a restart after upgrading packages, restart the runtime before continuing.


In [ ]:
!pip install -U openai openai-agents -q


## 2. Imports and API key


In [ ]:
from datetime import datetime, timedelta
from dataclasses import dataclass
from typing import Literal

from google.colab import userdata

import os

from agents import (
    Agent,
    Runner,
    WebSearchTool,
    ModelSettings,
    ItemHelpers,
    TResponseInputItem,
)

OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY


## 3. Searcher and evaluator agents

The important change is that Reuters is enforced at the **tool level** with `allowed_domains=["reuters.com"]`, rather than relying only on prompting.


In [ ]:
today_date = datetime.now().strftime("%Y-%m-%d")
two_days_ago = (datetime.now() - timedelta(days=2)).strftime("%Y-%m-%d")

INSTRUCTIONS_NEWS_SEARCH = f"""
You are a financial news research agent.

Use web search to find genuine Reuters news articles relevant to the user's request.

DATE WINDOW
- Earliest allowed publication date: {two_days_ago}
- Latest allowed publication date: {today_date}

SEARCH RULES
- Search only Reuters. The web-search tool is already restricted to reuters.com.
- Perform multiple searches with different keywords if the first search is insufficient.
- Search by topic synonyms, company names, sectors and relevant subtopics when useful.
- Do not substitute stock prices, ETF quotes, index values or generic market data for news articles.
- Do not invent headlines, dates, facts or URLs.
- Do not claim that there are no matching Reuters articles after only one narrow search.

OUTPUT RULES
- Respect the exact number of news items requested by the user.
- If the user does not specify a number, return 5 items.
- For every item include:
  1. Headline
  2. Publication date
  3. Short summary
  4. Publisher: Reuters
  5. Complete direct Reuters article URL
- Keep each item separate and clearly numbered.
- Only include articles whose publication date is inside the date window above.
- If, after several distinct searches, fewer valid Reuters articles are available, return the valid articles found and explicitly state how many were found. Never fabricate missing items.
"""

web_news_searcher = Agent(
    name="web_news_searcher",
    instructions=INSTRUCTIONS_NEWS_SEARCH,
    tools=[
        WebSearchTool(
            filters={"allowed_domains": ["reuters.com"]},
            search_context_size="high",
            external_web_access=True,
        )
    ],
    model_settings=ModelSettings(
        tool_choice="required",
    ),
)


@dataclass
class EvaluationFeedback:
    feedback: str
    score: Literal["successful", "unsuccessful"]


INSTRUCTIONS_NEWS_EVALUATOR = f"""
You are a strict evaluator of a Reuters financial-news search result.

You will receive BOTH:
1. the ORIGINAL USER REQUEST, and
2. the NEWS SUMMARY generated by the search agent.

Evaluate the result against the original request.

A result is successful only if ALL applicable requirements are satisfied:
- It contains exactly the number of news items requested by the user.
  If the user did not request a number, expect 5.
- Every item is a genuine Reuters news article.
- Every item contains a headline, publication date, publisher and complete Reuters URL.
- Every publication date is between {two_days_ago} and {today_date}, inclusive.
- Every item is relevant to the topic and region requested by the user.
- Stock prices, ETF quotes, index levels and generic market-data snapshots are not news items.
- If a required Reuters URL is missing, the result is unsuccessful.

Do not invent requirements that are absent from the original user request.
For example, do not fail a result for a missing geographic region if the user did not request one.

Return:
- score = "successful" only when all applicable requirements are met.
- score = "unsuccessful" otherwise.

When unsuccessful, give specific, actionable feedback for a completely new search.
"""

news_evaluator = Agent(
    name="news_evaluator",
    instructions=INSTRUCTIONS_NEWS_EVALUATOR,
    output_type=EvaluationFeedback,
)


## 4. Evaluator-Optimizer loop

Each retry starts from the **original request + evaluator feedback**. It does not reuse the previous failed news summary as authoritative context.


In [ ]:
async def main() -> None:
    msg = input("User's request: " ).strip()

    max_iterations = 4
    latest_outline: str = ""
    evaluator_feedback: str | None = None

    for iteration in range(1, max_iterations + 1):
        if evaluator_feedback is None:
            search_input: list[TResponseInputItem] = [
                {"content": msg, "role": "user"}
            ]
        else:
            search_input = [
                {"content": msg, "role": "user"},
                {
                    "content": (
                        "The previous attempt failed evaluation.\n\n"
                        f"Evaluator feedback:\n{evaluator_feedback}\n\n"
                        "Perform a NEW web search from scratch. "
                        "Do not reuse unsupported claims from the previous answer. "
                        "Correct every issue identified by the evaluator."
                    ),
                    "role": "user",
                },
            ]

        news_searcher_result = await Runner.run(
            web_news_searcher,
            search_input,
        )

        latest_outline = ItemHelpers.text_message_outputs(
            news_searcher_result.new_items
        )

        print(
            "\n\033[92m"
            f"************************** NEWS SEARCH {iteration} **************************"
            "\033[0m"
        )
        print(latest_outline)

        evaluator_input = (
            f"ORIGINAL USER REQUEST:\n{msg}\n\n"
            f"NEWS SUMMARY:\n{latest_outline}"
        )

        print(
            "\n\033[92m"
            "************************** RUNNING EVALUATION **************************"
            "\033[0m"
        )

        news_evaluator_result = await Runner.run(
            news_evaluator,
            evaluator_input,
        )
        result: EvaluationFeedback = news_evaluator_result.final_output

        print(f"\033[94mEvaluator score: {result.score}\033[0m")
        print(f"\033[94mEvaluator feedback: {result.feedback}\033[0m")

        if result.score == "successful":
            print("\033[92mEvaluation successful ==> stopping iteration.\033[0m")
            break

        evaluator_feedback = result.feedback

        if iteration == max_iterations:
            print("\033[91mReached max_iterations ==> stopping iteration.\033[0m")

    print(
        "\n\033[92m"
        "************************** FINAL NEWS SET **************************"
        "\033[0m"
    )
    print(latest_outline)


## 5. Run

Example request:

`Give me the latest 5 Reuters articles from the last 2 days about OpenAI, Nvidia, Oracle, Adobe, AI infrastructure or AI investment in the United States.`


In [ ]:
await main()
